# Phase 4: RAG Chain Testing

This notebook tests the full RAG pipeline against the 8 research questions.
Make sure `run_ingestion.py` has been run before executing cells here.

In [ ]:
import os, sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

from src.rag.rag_chain import ask
from src.storage.metadata_store import (
    get_theme_frequency, get_sentiment_distribution,
    get_source_distribution, get_all_unmet_needs, get_review_count
)

In [ ]:
# Database summary stats
print(f'Total reviews in DB: {get_review_count()}')
print(f'Sentiment distribution: {get_sentiment_distribution()}')
print(f'Source distribution: {get_source_distribution()}')

In [ ]:
# Top themes
import pandas as pd
theme_freq = get_theme_frequency()
df_themes = pd.DataFrame(theme_freq.items(), columns=['Theme', 'Count']).head(20)
df_themes

In [ ]:
# Test RAG against the 8 research questions
research_questions = [
    "Why do users repeatedly buy from the same categories on Blinkit?",
    "What prevents users from exploring new categories on the app?",
    "How do users currently discover new products on quick-commerce apps?",
    "What role do habit and routine play in shopping behavior?",
    "What trust signals do users need before trying an unfamiliar category?",
    "What frustrations recur across Blinkit and competitor reviews?",
    "Which user segments show higher exploration tendency?",
    "What unmet needs or category gaps come up consistently?",
]

for i, q in enumerate(research_questions, 1):
    print(f"\nQ{i}: {q}")
    print("-" * 60)
    result = ask(q)
    print(result['answer'])
    print(f"   [Sources used: {len(result['sources'])}]")

In [ ]:
# Traceability check: sample 10 themes and confirm each links to at least one review
from src.storage.metadata_store import get_reviews_by_theme

top_themes = list(get_theme_frequency().keys())[:10]
print('Traceability check:')
for theme in top_themes:
    reviews = get_reviews_by_theme(theme, limit=1)
    status = '✅ Traceable' if reviews else '❌ NOT TRACEABLE'
    print(f'  {status}: "{theme}" -> {reviews[0]["text"][:60]}...' if reviews else f'  {status}: "{theme}"')